In [13]:
# Enable tab autocomplete
%config IPCompleter.greedy=True
# Autoload changed files
%reload_ext autoreload
# Configure to reload all modules before executing code
%autoreload 2

# imports
from datetime import datetime, time
from zoneinfo import ZoneInfo

import ipywidgets as widgets
from dateutil.relativedelta import relativedelta
from IPython.display import clear_output, display

from backtest import Backtest, Holding, Portfolio

# Dollars initially invested
initial_investment = 1_000_000.0

# USD to CAD exchange rate
usd_cad = 1.3

my_portfolios = [
    Portfolio(
        name="Sell",
        holdings=[
            Holding("AAPL", 4.9),
            Holding("BRKA", 1.9),
            Holding("GOOG", 5.5),
            Holding("AMZN", 2.5),
            Holding("TSLA", 0.8),
        ],
    ),
    Portfolio(
        name="To Buy",
        holdings=[
            Holding("MSFT", 8.51),
            Holding("AAPL", 8.81),
            Holding("TSLA", 7.78),
            Holding("BAC", 8.15),
            Holding("NVDA", 8.25),
            Holding("GOOGL", 7.52),
            Holding("QQQ", 7.49),
            Holding("VOO", 7.32),
            Holding("BRK-B", 6.54),
            Holding("JPM", 6.82),
            Holding("MU", 5.66),
            Holding("AMZN", 6.63),
            Holding("SHOP", 6.63),
            Holding("DRAM", 3.87),
        ],
    ),
]


# Collect data for this time period
def set_tz(dt):
    return datetime.combine(dt, time.min, tzinfo=ZoneInfo("America/New_York"))


start = set_tz(datetime.now() - relativedelta(years=3))
end = set_tz(datetime.now())

# 4. Standard Date and Benchmark Controls
start_date_picker = widgets.DatePicker(description="Start Date:", value=start, style={"description_width": "6em"})
end_date_picker = widgets.DatePicker(description="End Date:", value=end, style={"description_width": "6em"})


options = [(x.name, x) for x in my_portfolios + Backtest.DEFAULT_PORTFOLIOS]
portfolio1_dropdown = widgets.Dropdown(
    options=options,
    index=0,
    description="Portfolio1:",
    style={"description_width": "6em"},
)

options = [("-- Select Portfolio --", None)] + options
portfolio2_dropdown = widgets.Dropdown(
    options=options,
    index=0,
    description="Portfolio2:",
    style={"description_width": "6em"},
)

options = [(x.name, x) for x in Backtest.DEFAULT_BENCHMARK_PORTFOLIOS]
benchmark_dropdown = widgets.Dropdown(
    options=options,
    # value="^GSPC",
    index=0,
    description="Benchmark:",
    style={"description_width": "6em"},
)

run_button = widgets.Button(
    description="Run Backtest",
    button_style="success",
    icon="check",
    layout={"width": "150px", "margin": "0px 0px 0px 7em"},
)


ui_controls = widgets.VBox(
    [
        widgets.HBox([start_date_picker, end_date_picker]),
        widgets.HBox([portfolio1_dropdown, portfolio2_dropdown]),
        widgets.HBox([benchmark_dropdown, run_button]),
    ]
)

ui = widgets.HBox([ui_controls])
output_area = widgets.Output()


def run_backtest(portfolios, benchmark_portfolio, start, end, initial_investment, usd_cad, output_area):
    start = set_tz(start)
    end = set_tz(end)

    # Portfolio: Ticker, percentage of holdings, description
    with output_area:
        clear_output(wait=True)
        print("run_backtest")
        Backtest.run(portfolios, benchmark_portfolio, start, end)


# Link the button click to the runner function
run_button.on_click(
    lambda b: run_backtest(
        [portfolio1_dropdown.value, portfolio2_dropdown.value],
        benchmark_dropdown.value,
        start_date_picker.value,
        end_date_picker.value,
        initial_investment,
        usd_cad,
        output_area,
    )
)
display(ui, output_area)

Output()